# 📊 보행 임베딩 비교 분석 노트북
`scripts/build_embeddings.py`로 생성된 임베딩 결과를 시각화하고 성능을 비교합니다.

## 먼저 아래 명령 중 원하는 것을 실행하세요
```bash
# CNN + Master 데이터
python scripts/build_embeddings.py --input_path data/processed/Master_Gait_Dataset.parquet --model cnn

# CNN + Raw 데이터
python scripts/build_embeddings.py --input_path data/processed/raw_merged.parquet --model cnn

# AutoEncoder 실행
python scripts/build_embeddings.py --input_path data/processed/Master_Gait_Dataset.parquet --model ae

# Centroid 실행
python scripts/build_embeddings.py --input_path data/processed/Master_Gait_Dataset.parquet --model centroid
```


In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import silhouette_score

# 경로 설정
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parent
PROCESSED_DIR = ROOT / "data" / "processed"

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print(f"ROOT: {ROOT}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")


In [ ]:
def find_embedding_files():
    return sorted(PROCESSED_DIR.glob("embedding_results_*.parquet"))

files = find_embedding_files()
print(f"발견된 임베딩 결과 파일: {len(files)}개")
for f in files:
    print(f"  - {f.name}")


In [ ]:
def load_embedding(file_path):
    df = pd.read_parquet(file_path)
    print(f"✅ {file_path.name}")
    print(f"   {df.shape[0]}개 Trial | 그룹 분포: {df['group'].value_counts().to_dict()}")
    return df


In [ ]:
GROUP_STYLE = {
    "ACLD":                {"color": "#E63946", "marker": "o"},
    "ACLR":                {"color": "#F4A261", "marker": "s"},
    "Healthy adults":      {"color": "#2A9D8F", "marker": "^"},
    "Healthy adolescents": {"color": "#457B9D", "marker": "D"},
}
DEFAULT_STYLE = {"color": "#999999", "marker": "x"}


In [ ]:
def plot_embedding(df, title="", ax=None):
    """1개의 임베딩 결과 산점도 및 Silhouette Score 출력"""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(9, 7))

    for group, gdf in df.groupby("group"):
        style = GROUP_STYLE.get(group, DEFAULT_STYLE)
        ax.scatter(gdf["umap_x"], gdf["umap_y"],
                   c=style["color"], marker=style["marker"],
                   label=group, alpha=0.75, s=55,
                   edgecolors="white", linewidths=0.4)

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("UMAP Dim 1")
    ax.set_ylabel("UMAP Dim 2")
    ax.legend(fontsize=9, framealpha=0.8)
    ax.grid(True, alpha=0.2)

    # Silhouette Score
    emb_cols = [c for c in df.columns if c.startswith("emb_")]
    if emb_cols:
        groups = df["group"].unique()
        label_map = {g: i for i, g in enumerate(groups)}
        y = np.array([label_map[g] for g in df["group"]])
        X = df[emb_cols].values
        score = silhouette_score(X, y, sample_size=min(1000, len(df)), random_state=42)
        ax.text(0.02, 0.98, f"Silhouette: {score:.3f}",
                transform=ax.transAxes, fontsize=9, va="top", ha="left",
                bbox=dict(boxstyle="round", fc="white", alpha=0.7))
        if standalone:
            plt.tight_layout()
            plt.show()
        return score

    if standalone:
        plt.tight_layout()
        plt.show()
    return None


In [ ]:
def compare_embeddings(file_a, file_b, label_a="A", label_b="B"):
    """두 임베딩 결과를 나란히 비교 시각화"""
    df_a = load_embedding(Path(file_a))
    df_b = load_embedding(Path(file_b))

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("보행 임베딩 결과 비교 분석", fontsize=14, fontweight="bold")

    score_a = plot_embedding(df_a, label_a, ax=axes[0])
    score_b = plot_embedding(df_b, label_b, ax=axes[1])

    plt.tight_layout()
    plt.show()

    if score_a is not None and score_b is not None:
        print(f"\n📊 Silhouette Score 비교 (높을수록 군집 분리 우수)")
        print(f"   {label_a:<40}: {score_a:.4f}")
        print(f"   {label_b:<40}: {score_b:.4f}")
        winner = label_a if score_a >= score_b else label_b
        print(f"   🏆 우승: {winner}")


In [ ]:
def plot_subject(df, subject_id):
    """특정 피험자의 Trial들이 UMAP 공간에서 어떻게 분포하는지 확인"""
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(df["umap_x"], df["umap_y"], c="#cccccc", alpha=0.3, s=20, label="Others")

    sub = df[df["subject_id"] == subject_id]
    if sub.empty:
        print(f"'{subject_id}' 데이터 없음")
        return

    ax.scatter(sub["umap_x"], sub["umap_y"],
               c="#E63946", s=120, marker="*",
               edgecolors="black", linewidths=0.5,
               label=f"{subject_id} ({len(sub)} trials)")
    ax.set_title(f"피험자 [{subject_id}] Trial 분포", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()


## 🚀 실행 예시

### Master vs Raw 교차 비교
```python
compare_embeddings(
    file_a=PROCESSED_DIR / "embedding_results_cnn_Master_Gait_Dataset.parquet",
    file_b=PROCESSED_DIR / "embedding_results_cnn_raw_merged.parquet",
    label_a="CNN / Master (315 피처, Feature Selected)",
    label_b="CNN / Raw (693 피처, 전체)",
)
```

### 단일 결과 시각화
```python
df = load_embedding(PROCESSED_DIR / "embedding_results_cnn_Master_Gait_Dataset.parquet")
plot_embedding(df, title="CNN Embedding - Master Dataset")
```

### 특정 피험자 분포 확인
```python
df = load_embedding(PROCESSED_DIR / "embedding_results_cnn_Master_Gait_Dataset.parquet")
plot_subject(df, subject_id="ACLD3")
```
